# 08. 에이전트 하네스

## 학습 목표
- AgentHarness의 개념과 역할을 이해한다
- 하네스의 핵심 기능(계획, 파일시스템, 태스크 위임)을 안다
- 컨텍스트 관리(오프로딩, 요약)를 이해한다
- 코드 실행과 Human-in-the-Loop을 설정한다
- 스킬과 메모리 시스템을 연동한다

In [ ]:
# 환경 설정
from dotenv import load_dotenv
import os

load_dotenv()
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY가 설정되지 않았습니다!"
print("환경 설정 완료")

In [ ]:
# Observability 설정 (선택) - LangSmith 또는 Langfuse
# .env에 키를 설정하거나, 아래 주석을 해제하여 직접 입력하세요.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: LANGSMITH_TRACING=true 시 자동 활성화 (코드 수정 불필요)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault(
        "LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default")
    )
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: invoke/stream 호출 시 config={"callbacks": [langfuse_handler]} 전달
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler

    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")
# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}

In [ ]:
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4.1")

print(f"모델 설정 완료: {model.model_name}")

---
## 1. AgentHarness 개념

**AgentHarness**는 장기 실행 자율 에이전트를 위한 **포괄적 기능 제공자**입니다.  
에이전트가 복잡한 멀티 스텝 작업을 수행할 때 필요한 모든 인프라를 하나로 묶어 제공합니다.

### 하네스가 제공하는 핵심 기능

| 기능 | 설명 |
|------|------|
| **Planning** | 구조화된 태스크 리스트 관리 (`write_todos`) |
| **Filesystem** | 가상/로컬 파일 읽기, 쓰기, 검색 |
| **Task Delegation** | 서브에이전트를 통한 작업 위임 |
| **Context Management** | 오프로딩 및 요약을 통한 컨텍스트 압축 |
| **Code Execution** | 샌드박스 환경에서 안전한 코드 실행 |
| **Human-in-the-Loop** | 민감 작업에 대한 사람 승인 |
| **Skills & Memory** | 전문 워크플로와 영속적 지식 |

`create_deep_agent()`를 호출하면 이 모든 기능이 자동으로 조립되어 하나의 에이전트로 제공됩니다.

In [ ]:
# AgentHarness 개념 — create_deep_agent가 하네스를 조립합니다
harness_config = {
    "model": "gpt-4.1",
    "system_prompt": "당신은 프로젝트 관리 어시스턴트입니다.",
    "planning": True,  # write_todos 도구 활성화
    "filesystem": True,  # 파일시스템 도구 활성화
    "subagents": [],  # 서브에이전트 목록
    "context_management": True,  # 컨텍스트 압축 활성화
}

print("AgentHarness 구성 요소:")
for key, value in harness_config.items():
    print(f"  {key}: {value}")

---
## 2. 계획 도구

에이전트는 `write_todos` 도구를 사용하여 복잡한 작업을 **구조화된 태스크 리스트**로 분해합니다.  
각 태스크는 상태를 가집니다:

| 상태 | 설명 |
|------|------|
| `pending` | 아직 시작하지 않음 |
| `in_progress` | 현재 진행 중 |
| `completed` | 완료됨 |

In [ ]:
# write_todos 도구 — 구조화된 태스크 리스트 예시
todo_list = [
    {"task": "프로젝트 구조 분석", "status": "completed"},
    {"task": "API 엔드포인트 설계", "status": "in_progress"},
    {"task": "데이터베이스 스키마 작성", "status": "pending"},
    {"task": "테스트 코드 작성", "status": "pending"},
    {"task": "문서화", "status": "pending"},
]

print("=== 에이전트 태스크 리스트 ===")
for i, item in enumerate(todo_list, 1):
    icon = {"completed": "[x]", "in_progress": "[-]", "pending": "[ ]"}
    print(f"  {icon[item['status']]} {i}. {item['task']}")

---
## 3. 가상 파일시스템

하네스는 구성 가능한 파일시스템 백엔드를 통해 표준 파일 작업을 지원합니다.

| 도구 | 설명 |
|------|------|
| `ls` | 디렉토리 목록 (메타데이터 포함) |
| `read_file` | 파일 내용 읽기 (줄 번호 포함, 이미지 지원) |
| `write_file` | 파일 생성 |
| `edit_file` | 문자열 치환 편집 |
| `glob` | 패턴 기반 파일 검색 |
| `grep` | 내용 검색 (여러 출력 모드) |
| `execute` | 쉘 명령 실행 (샌드박스 백엔드 전용) |

In [ ]:
# 파일시스템 도구 사용 예시 (참고용)
fs_operations = {
    "ls": 'ls(path="/project/src")',
    "read_file": 'read_file(path="/project/src/main.py")',
    "write_file": 'write_file(path="/project/config.yaml", content="debug: true")',
    "edit_file": 'edit_file(path="/project/src/main.py", old="v1", new="v2")',
    "glob": 'glob(pattern="**/*.py")',
    "grep": 'grep(pattern="TODO", path="/project/src")',
}

print("=== 파일시스템 도구 호출 예시 ===")
for tool_name, call_example in fs_operations.items():
    print(f"  {tool_name:12s} -> {call_example}")

---
## 4. 태스크 위임 — 서브에이전트

하네스는 메인 에이전트가 **임시 서브에이전트**를 생성하여 격리된 멀티 스텝 태스크를 수행할 수 있게 합니다.

### 서브에이전트의 장점

| 장점 | 설명 |
|------|------|
| **컨텍스트 격리** | 서브에이전트 실행이 메인 컨텍스트를 오염시키지 않음 |
| **병렬 실행** | 여러 서브에이전트를 동시에 실행 가능 |
| **전문화** | 각 서브에이전트에 특화된 도구와 프롬프트 제공 |
| **토큰 효율** | 결과 압축으로 메인 에이전트의 토큰 절약 |

In [ ]:
# 서브에이전트 위임 구성 예시 (참고용)
subagent_config = [
    {
        "name": "researcher",
        "description": "인터넷 검색으로 정보를 조사합니다.",
        "system_prompt": "검색 결과를 간결하게 요약하세요.",
        "tools": ["internet_search"],
    },
    {
        "name": "coder",
        "description": "코드를 작성하고 테스트합니다.",
        "system_prompt": "깔끔하고 테스트 가능한 코드를 작성하세요.",
        "tools": ["write_file", "execute"],
    },
]

print("=== 서브에이전트 구성 ===")
for sa in subagent_config:
    print(f"  [{sa['name']}] {sa['description']}")
    print(f"    도구: {', '.join(sa['tools'])}")

---
## 5. 컨텍스트 관리

장기 실행 에이전트의 가장 큰 과제는 **컨텍스트 윈도우 한계**입니다.  
하네스는 두 가지 기법으로 이를 해결합니다.

### 입력 컨텍스트 조립
시스템 프롬프트, 지침, 메모리 가이드라인, 스킬 정보, 파일시스템 문서를 종합하여 초기 프롬프트를 구성합니다.

### 런타임 컨텍스트 압축

| 기법 | 동작 | 트리거 |
|------|------|--------|
| **오프로딩** | 20,000 토큰 초과 콘텐츠를 디스크에 저장, 포인터 참조 유지 | 콘텐츠 크기 기준 |
| **요약** | 대화 히스토리를 구조화된 요약으로 압축 | 모델 윈도우 한계 접근 시 |

원본 메시지는 파일시스템 스토리지에 보존되므로 정보 손실이 없습니다.

In [ ]:
# 컨텍스트 관리 설정 예시 (참고용)
context_config = {
    "offloading": {
        "enabled": True,
        "threshold_tokens": 20000,
        "storage": "filesystem",
    },
    "summarization": {
        "enabled": True,
        "trigger": "window_limit_approach",
        "preserve_original": True,
    },
}

print("=== 컨텍스트 관리 설정 ===")
for section, settings in context_config.items():
    print(f"\n[{section}]")
    for key, value in settings.items():
        print(f"  {key}: {value}")

---
## 6. 코드 실행

샌드박스 백엔드는 `execute` 도구를 노출하여 격리된 환경에서 명령을 실행합니다.  
호스트 시스템에 영향을 주지 않으면서 보안성, 깨끗한 환경, 재현성을 제공합니다.

In [ ]:
# 샌드박스 코드 실행 예시 (참고용)
execute_examples = [
    {"command": "python -c 'print(2+2)'", "desc": "Python 코드 실행"},
    {"command": "pip install requests", "desc": "패키지 설치"},
    {"command": "pytest tests/", "desc": "테스트 실행"},
]

print("=== 샌드박스 execute 도구 예시 ===")
for ex in execute_examples:
    print(f"  $ {ex['command']}")
    print(f"    -> {ex['desc']}")

---
## 7. Human-in-the-Loop

선택적 인터럽트 설정으로 지정된 도구 호출 시 사람의 승인을 요구합니다.

In [ ]:
# Human-in-the-Loop 설정 예시 (참고용)
hitl_config = {
    "interrupt_on": {
        "write_file": True,  # 파일 쓰기 전 승인
        "edit_file": True,  # 파일 편집 전 승인
        "execute": True,  # 명령 실행 전 승인
    }
}

print("=== Human-in-the-Loop 설정 ===")
print("승인이 필요한 도구:")
for tool, enabled in hitl_config["interrupt_on"].items():
    status = "승인 필요" if enabled else "자동 실행"
    print(f"  {tool}: {status}")

print("\n승인 옵션: approve(승인), reject(거부), edit(수정)")

---
## 8. 스킬과 메모리

### 스킬 (Skills)
**Agent Skills 표준**을 따르는 전문 워크플로입니다.  
관련성이 있을 때 점진적으로 로드되어 토큰 소비를 줄입니다.

- 각 스킬은 `SKILL.md` 파일로 정의
- 트리거 조건에 따라 자동 활성화
- 도구, 프롬프트, 워크플로를 캡슐화

### 메모리 (Memory)
**AGENTS.md** 형식의 영속적 컨텍스트 파일입니다.  
대화를 넘어서 재사용 가능한 가이드라인, 선호도, 프로젝트 지식을 제공합니다.

| 구분 | 위치 | 범위 |
|------|------|------|
| 글로벌 메모리 | `~/.deepagents/<agent>/memories/` | 모든 프로젝트 |
| 프로젝트 메모리 | `.deepagents/AGENTS.md` | 현재 프로젝트 |

In [ ]:
# 스킬과 메모리 설정 예시 (참고용)
skills_config = [
    {"name": "code-review", "trigger": "코드 리뷰 요청 시"},
    {"name": "test-writer", "trigger": "테스트 작성 요청 시"},
    {"name": "doc-generator", "trigger": "문서화 요청 시"},
]

memory_config = {
    "global": "~/.deepagents/my-agent/memories/",
    "project": ".deepagents/AGENTS.md",
}

print("=== 스킬 설정 ===")
for skill in skills_config:
    print(f"  [{skill['name']}] 트리거: {skill['trigger']}")

print("\n=== 메모리 설정 ===")
for scope, path in memory_config.items():
    print(f"  {scope}: {path}")

---
## 전체 요약

| 주제 | 핵심 개념 | 핵심 API/도구 |
|------|----------|-------------|
| 하네스 개념 | 장기 실행 에이전트를 위한 포괄적 기능 제공자 | `create_deep_agent()` |
| 계획 도구 | 구조화된 태스크 리스트 관리 | `write_todos` |
| 파일시스템 | 가상/로컬 파일 작업 (7종 도구) | `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`, `execute` |
| 서브에이전트 | 격리된 태스크 위임, 병렬 실행 | `subagents`, `task` |
| 컨텍스트 관리 | 오프로딩(20K 토큰), 요약 압축 | 자동 관리 |
| 코드 실행 | 샌드박스에서 안전한 명령 실행 | `execute` |
| HITL | 민감 도구 호출 시 사람 승인 | `interrupt_on` |
| 스킬/메모리 | 전문 워크플로 + 영속적 컨텍스트 | `SKILL.md`, `AGENTS.md` |

### 다음 단계
→ **[09. 외부 프레임워크 비교](09_comparison.ipynb)**으로 진행하세요!

---
**참고 문서:**
- [Deep Agents Harness](../docs/deepagents/05-harness.md)